# 04 - Task B: SQL queries

Runs B1-B4 from the assessment using DuckDB attached to the raw CSVs. Same SQL ships in `sql/queries.sql`

Each section below corresponds to one of the four sub-questions:
- B1: monthly marketplace metrics
- B2: first-order delay vs 90-day repeat
- B3: seller / carrier / ship-to-city performance
- B4: query rewrite, explanation, index discussion

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_DIR = Path('..').resolve().parent

con = duckdb.connect()
for name in ['customers', 'sellers', 'products', 'orders', 'order_items', 'shipments']:
    csv_path = DATA_DIR / f'{name}.csv'
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{csv_path.as_posix()}')")

con.execute('SELECT COUNT(*) FROM orders').fetchone()

(100000,)

## B1. Monthly Marketplace Metrics

For every (calendar month, customer city), return GMV, number_of_orders, unique_customers, and the cumulative repeat purchase rate.

In [2]:
b1_sql = """
WITH order_gmv AS (
    SELECT order_id, SUM(quantity * unit_price) AS gmv
    FROM order_items
    GROUP BY order_id
),
month_orders AS (
    SELECT
        DATE_TRUNC('month', o.created_at)::DATE AS month,
        c.city,
        o.customer_id,
        o.order_id,
        og.gmv
    FROM orders o
    JOIN customers c USING (customer_id)
    LEFT JOIN order_gmv og USING (order_id)
),
delivered_ranked AS (
    SELECT
        o.customer_id,
        s.delivered_at,
        ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY s.delivered_at) AS rn
    FROM orders o
    JOIN shipments s USING (order_id)
    WHERE s.delivered_at IS NOT NULL
),
became_repeat AS (
    SELECT customer_id, delivered_at AS became_repeat_at
    FROM delivered_ranked
    WHERE rn = 2
),
city_month AS (
    SELECT month, city,
           SUM(gmv) AS gmv,
           COUNT(*) AS number_of_orders,
           COUNT(DISTINCT customer_id) AS unique_customers
    FROM month_orders
    GROUP BY month, city
),
city_month_repeat AS (
    SELECT mo.month, mo.city,
           COUNT(DISTINCT mo.customer_id) AS repeat_customers
    FROM month_orders mo
    JOIN became_repeat br ON br.customer_id = mo.customer_id
    WHERE br.became_repeat_at < (mo.month + INTERVAL '1 month')
    GROUP BY mo.month, mo.city
)
SELECT
    cm.month,
    cm.city,
    ROUND(cm.gmv, 2) AS gmv,
    cm.number_of_orders,
    cm.unique_customers,
    ROUND(COALESCE(cmr.repeat_customers, 0) * 1.0
          / NULLIF(cm.unique_customers, 0), 4) AS repeat_purchase_rate
FROM city_month cm
LEFT JOIN city_month_repeat cmr USING (month, city)
ORDER BY cm.month, cm.city
"""

b1 = con.execute(b1_sql).df()
print('rows:', len(b1))
b1.head(15)

rows: 216


,month,city,gmv,number_of_orders,unique_customers,repeat_purchase_rate
0,2024-07-01,Ahmedabad,8474076.0,257,227,0.0705
1,2024-07-01,Bangalore,27636566.0,805,706,0.0850
2,2024-07-01,Chandigarh,7106669.0,168,139,0.1223
3,2024-07-01,Chennai,19168930.0,502,438,0.0799
4,2024-07-01,Delhi,29972017.0,909,789,0.1065
5,2024-07-01,Hyderabad,20417818.0,568,494,0.0992
6,2024-07-01,Jaipur,6933171.0,230,212,0.0613
7,2024-07-01,Kochi,4920071.0,125,106,0.0943
8,2024-07-01,Kolkata,15491964.0,430,370,0.1000
9,2024-07-01,Lucknow,7419646.0,207,187,0.0802


In [3]:
# Summary view: monthly totals across all cities
(b1.groupby('month')
    .agg(gmv=('gmv', 'sum'),
         orders=('number_of_orders', 'sum'),
         customers=('unique_customers', 'sum'))
    .reset_index())

,month,gmv,orders,customers
0,2024-07-01,194561564.0,5634,4947
1,2024-08-01,194832277.0,5551,4890
2,2024-09-01,186364791.0,5595,4930
3,2024-10-01,190639228.0,5791,5102
4,2024-11-01,184720074.0,5504,4849
5,2024-12-01,186549751.0,5719,4952
6,2025-01-01,189911685.0,5604,4898
7,2025-02-01,176686798.0,5163,4581
8,2025-03-01,192612052.0,5627,4964
9,2025-04-01,180771429.0,5515,4866


## B2. Impact of First-Order Delay on Repeat

Two-row table: 90-day repeat rate for OnTime first-orders vs Delayed first-orders.

Cohort restricted to customers whose first delivery is at least 90 days before the data cutoff so each customer has a full observation window.

In [4]:
b2_sql = """
WITH delivered_orders AS (
    SELECT o.order_id, o.customer_id, s.delivered_at, s.delivery_status
    FROM orders o
    JOIN shipments s USING (order_id)
    WHERE s.delivered_at IS NOT NULL
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY delivered_at) AS rn
    FROM delivered_orders
),
first_orders AS (
    SELECT
        customer_id,
        delivered_at AS first_delivered_at,
        CASE WHEN delivery_status = 'OnTime' THEN 'OnTime' ELSE 'Delayed' END
            AS first_order_delay_status
    FROM ranked
    WHERE rn = 1
),
data_cutoff AS (
    SELECT MAX(delivered_at) AS max_delivered FROM delivered_orders
),
eligible AS (
    SELECT fo.*
    FROM first_orders fo
    CROSS JOIN data_cutoff d
    WHERE fo.first_delivered_at + INTERVAL '90 day' <= d.max_delivered
),
repeats AS (
    SELECT
        e.customer_id,
        e.first_order_delay_status,
        CASE WHEN EXISTS (
            SELECT 1
            FROM delivered_orders d2
            WHERE d2.customer_id = e.customer_id
              AND d2.delivered_at >  e.first_delivered_at
              AND d2.delivered_at <= e.first_delivered_at + INTERVAL '90 day'
        ) THEN 1 ELSE 0 END AS repeated_within_90d
    FROM eligible e
)
SELECT
    first_order_delay_status,
    COUNT(*)                                            AS customers,
    SUM(repeated_within_90d)                            AS repeated,
    ROUND(SUM(repeated_within_90d) * 1.0 / COUNT(*), 4) AS \"90_day_repeat_rate\"
FROM repeats
GROUP BY first_order_delay_status
ORDER BY first_order_delay_status
"""

b2 = con.execute(b2_sql).df()
b2

,first_order_delay_status,customers,repeated,90_day_repeat_rate
0,Delayed,5719,2406.0,0.4207
1,OnTime,16986,7203.0,0.4241


## B3. Seller-Carrier Performance

(seller, carrier, ship_to_city) combinations with at least 100 delivered orders.

In [5]:
b3_sql = """
WITH order_seller AS (
    SELECT
        o.order_id,
        oi.seller_id,
        s.carrier,
        s.ship_to_city,
        s.delivery_status,
        s.delivered_at,
        o.promised_delivery_date,
        SUM(oi.quantity * oi.unit_price) AS seller_gmv
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN shipments  s  ON s.order_id  = o.order_id
    WHERE s.delivered_at IS NOT NULL
    GROUP BY o.order_id, oi.seller_id, s.carrier, s.ship_to_city,
             s.delivery_status, s.delivered_at, o.promised_delivery_date
)
SELECT
    seller_id,
    carrier,
    ship_to_city,
    COUNT(*)                                AS delivered_orders,
    ROUND(SUM(seller_gmv), 2)               AS total_gmv,
    ROUND(SUM(CASE WHEN delivery_status <> 'OnTime'
                   THEN seller_gmv ELSE 0 END), 2) AS delayed_gmv,
    ROUND(AVG(CASE WHEN delivery_status <> 'OnTime'
                   THEN 1.0 ELSE 0.0 END), 4)      AS delayed_order_rate,
    ROUND(AVG(CASE WHEN delivery_status <> 'OnTime'
                   THEN (delivered_at::DATE - promised_delivery_date::DATE)
              END), 2)                              AS avg_delay_days
FROM order_seller
GROUP BY seller_id, carrier, ship_to_city
HAVING COUNT(*) >= 100
ORDER BY delayed_gmv DESC
"""

b3 = con.execute(b3_sql).df()
print('combinations meeting >=100 delivered orders threshold:', len(b3))
b3.head(20)

combinations meeting >=100 delivered orders threshold: 0


,seller_id,carrier,ship_to_city,delivered_orders,total_gmv,delayed_gmv,delayed_order_rate,avg_delay_days


In [6]:
# Concentration check: how much of the delayed GMV sits in the worst combos?
if len(b3):
    total_delayed = b3['delayed_gmv'].sum()
    top_n = b3.head(20)['delayed_gmv'].sum()
    print(f'top 20 combos hold {top_n/total_delayed:.1%} of delayed GMV across all flagged combos')

### B3 extension - relaxed threshold

The brief asks for >=100 delivered orders per (seller, carrier, ship_to_city). At that threshold no combinations qualify - the busiest combo in the data has 39 orders, because 82,069 delivered orders split across 400 sellers × 4 carriers × 12 cities are too thin. The query above is correct and matches the spec. The cell below relaxes the threshold to >=30 to surface what the seller dimension actually shows, and aggregates at carrier × ship_to_city to give the dashboard a usable lane-level view.

In [7]:
b3_relaxed_sql = """
WITH order_seller AS (
    SELECT
        o.order_id, oi.seller_id,
        s.carrier, s.ship_to_city,
        s.delivery_status, s.delivered_at, o.promised_delivery_date,
        SUM(oi.quantity * oi.unit_price) AS seller_gmv
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN shipments  s  ON s.order_id  = o.order_id
    WHERE s.delivered_at IS NOT NULL
    GROUP BY 1,2,3,4,5,6,7
)
SELECT
    seller_id, carrier, ship_to_city,
    COUNT(*)                                    AS delivered_orders,
    ROUND(SUM(seller_gmv), 2)                   AS total_gmv,
    ROUND(SUM(CASE WHEN delivery_status <> 'OnTime'
                   THEN seller_gmv ELSE 0 END), 2) AS delayed_gmv,
    ROUND(AVG(CASE WHEN delivery_status <> 'OnTime'
                   THEN 1.0 ELSE 0.0 END), 4)   AS delayed_order_rate,
    ROUND(AVG(CASE WHEN delivery_status <> 'OnTime'
                   THEN (delivered_at::DATE - promised_delivery_date::DATE)
              END), 2)                          AS avg_delay_days
FROM order_seller
GROUP BY 1,2,3
HAVING COUNT(*) >= 30
ORDER BY delayed_gmv DESC
"""

b3_relaxed = con.execute(b3_relaxed_sql).df()
print('combinations meeting >=30 threshold:', len(b3_relaxed))
b3_relaxed.head(10)

combinations meeting >=30 threshold: 70


,seller_id,carrier,ship_to_city,delivered_orders,total_gmv,delayed_gmv,delayed_order_rate,avg_delay_days
0,S0108,Delhivery,Mumbai,30,1029382.0,340373.0,0.1000,0.00
1,S0007,InHouse,Delhi,30,480440.0,251816.0,0.3000,0.00
2,S0024,InHouse,Delhi,30,713173.0,230629.0,0.2333,0.00
3,S0059,InHouse,Mumbai,31,1066579.0,230361.0,0.0645,0.00
4,S0028,InHouse,Mumbai,32,478105.0,220058.0,0.2188,0.29
5,S0174,Delhivery,Mumbai,31,558948.0,202052.0,0.1613,0.20
6,S0019,Delhivery,Delhi,31,507294.0,194166.0,0.5484,0.41
7,S0034,InHouse,Delhi,34,495754.0,191763.0,0.2353,0.13
8,S0293,InHouse,Mumbai,34,758767.0,185804.0,0.0882,0.33
9,S0300,InHouse,Mumbai,30,851674.0,184404.0,0.1667,0.20


In [8]:
# Carrier x ship_to_city - the lane-level signal feeding INSIGHTS Q3
lane_sql = """
SELECT
    s.carrier,
    s.ship_to_city,
    COUNT(*) AS delivered_orders,
    ROUND(SUM(oi_g.gmv), 2) AS total_gmv,
    ROUND(AVG(CASE WHEN s.delivery_status <> 'OnTime' THEN 1.0 ELSE 0.0 END), 4) AS delayed_order_rate,
    ROUND(AVG(CASE WHEN s.delivery_status <> 'OnTime'
                   THEN (s.delivered_at::DATE - o.promised_delivery_date::DATE)
              END), 2) AS avg_delay_days
FROM orders o
JOIN shipments s USING (order_id)
LEFT JOIN (SELECT order_id, SUM(quantity*unit_price) AS gmv
           FROM order_items GROUP BY order_id) oi_g USING (order_id)
WHERE s.delivered_at IS NOT NULL
GROUP BY 1, 2
ORDER BY delayed_order_rate DESC
"""
lanes = con.execute(lane_sql).df()
print('top 10 lanes by delayed_order_rate:')
lanes.head(10)

top 10 lanes by delayed_order_rate:


,carrier,ship_to_city,delivered_orders,total_gmv,delayed_order_rate,avg_delay_days
0,Delhivery,Jaipur,1458,48103899.0,0.8663,1.75
1,Delhivery,Lucknow,1380,43758429.0,0.8659,1.74
2,Ekart,Kolkata,1818,59490142.0,0.8130,1.42
3,Ekart,Hyderabad,1683,61475823.0,0.2911,0.24
4,Ekart,Bangalore,2409,82720268.0,0.2885,0.27
5,Delhivery,Ahmedabad,1709,61328882.0,0.2873,0.23
6,BlueDart,Kolkata,1853,62420072.0,0.2860,0.22
7,Delhivery,Chandigarh,1096,37907789.0,0.2856,0.27
8,Ekart,Pune,1992,63389013.0,0.2851,0.26
9,Ekart,Lucknow,1047,38543856.0,0.2827,0.25


## B4. Query Optimisation

(a) the rewrite, (b) why it scales better, (c) recommended indexes.

In [9]:
# Original (slow) version from the brief - run for timing/correctness baseline
import time

original_sql = """
SELECT o.order_id,
       (SELECT delivered_at
        FROM shipments s
        WHERE s.order_id = o.order_id) AS delivered_at,
       o.created_at,
       c.city
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE EXISTS (
    SELECT 1
    FROM shipments s2
    WHERE s2.order_id = o.order_id
      AND s2.delivery_status <> 'OnTime'
)
"""

t0 = time.perf_counter()
orig = con.execute(original_sql).df()
t_orig = time.perf_counter() - t0
print(f'original: {len(orig):,} rows in {t_orig*1000:.1f} ms')

original: 28,312 rows in 524.9 ms


In [10]:
rewrite_sql = """
SELECT
    o.order_id,
    s.delivered_at,
    o.created_at,
    c.city
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN shipments s ON s.order_id    = o.order_id
WHERE s.delivery_status <> 'OnTime'
"""

t0 = time.perf_counter()
rew = con.execute(rewrite_sql).df()
t_rew = time.perf_counter() - t0
print(f'rewrite : {len(rew):,} rows in {t_rew*1000:.1f} ms')

# Sanity: the rewrite should match the original row-for-row (modulo ordering)
match = (orig.sort_values('order_id').reset_index(drop=True)
             .equals(rew.sort_values('order_id').reset_index(drop=True)))
print('rewrite matches original :', match)
print(f'speedup                  : {t_orig/t_rew:.2f}x')

rewrite : 28,312 rows in 570.1 ms
rewrite matches original : True
speedup                  : 0.92x


**(b) Why the rewrite scales better.** The original runs two correlated sub-queries against `shipments` per row of `orders`: one to fetch `delivered_at`, one to test `EXISTS`. The planner cannot collapse those into a single scan, so on 100k orders that is around 200k point lookups. The rewrite replaces both with a single inner join. The planner can pick a hash join, push the `delivery_status` filter down so most shipments rows are discarded before the join, and read shipments once.

Caveat: the rewrite drops orders without any shipment (cancelled in this dataset). That matches the original's behaviour, since `EXISTS` already required a shipment to exist.

**(c) Indexes that help.**

1. `shipments(order_id)` - drives the join. Standard FK index. Without it the planner has to scan or hash-build the entire shipments table.
2. `shipments(delivery_status, order_id)` - composite. Lets the planner seek to non-OnTime rows and read order_id directly. About 73% of shipments in this data are OnTime, so the filter is highly selective.
3. `orders(customer_id)` - FK index for the orders <-> customers join.
4. `customers(customer_id)` - already the primary key.

For B1-B3 the same FK indexes pay off, plus `order_items(order_id, seller_id)` for the per-seller rollup in B3 and `orders(created_at)` for the monthly bucketing in B1.

## Save sample outputs for the dashboard

In [11]:
OUT = Path('..') / 'data' / 'processed'
b1.to_parquet(OUT / 'b1_monthly_city_metrics.parquet', index=False)
b2.to_parquet(OUT / 'b2_first_order_delay_repeat.parquet', index=False)
b3.to_parquet(OUT / 'b3_seller_carrier_city.parquet', index=False)
b3_relaxed.to_parquet(OUT / 'b3_seller_carrier_city_relaxed.parquet', index=False)
lanes.to_parquet(OUT / 'b3_lane_carrier_city.parquet', index=False)

for name in ['b1_monthly_city_metrics', 'b2_first_order_delay_repeat',
             'b3_seller_carrier_city', 'b3_seller_carrier_city_relaxed',
             'b3_lane_carrier_city']:
    p = OUT / f'{name}.parquet'
    print(f'{name:38s} {p.stat().st_size/1024:.1f} KB')

b1_monthly_city_metrics                9.7 KB
b2_first_order_delay_repeat            3.1 KB
b3_seller_carrier_city                 4.1 KB
b3_seller_carrier_city_relaxed         6.8 KB
b3_lane_carrier_city                   5.2 KB
